# lora-1.7b-malay50h-babel on Colab

Resumes the Malay LoRA run (malay_50h_mix + BabelSpeech, 83h, 5,098 steps) from the checkpoint on Google Drive,
then evaluates. Everything persistent lives in Drive under `My_dataset/`:

- `malay_50h_babel/` - prepared data (train / valid / test / cs_test / babel_test)
- `outputs/lora-1.7b-malay50h-babel/` - checkpoints (resume point), final adapter, train log
- `outputs/lora-1.7b-malay50h-mix/adapter` - previous best, for the babel_test comparison

`outputs/` in the repo is a symlink to that Drive folder, so a disconnect loses at most 200 steps:
rerun all cells and training resumes from the latest checkpoint.

Needs a bf16 GPU (L4 / A100). A T4 does not support bf16.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "need a bf16 GPU (L4 / A100), not T4/CPU"

## 1. Drive, code, packages

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/My_dataset"
!ls "$DRIVE" "$DRIVE/outputs/lora-1.7b-malay50h-babel"

In [ ]:
%cd /content
!git -C finetune_QwenASR pull -q 2>/dev/null || git clone -q https://github.com/duclvq/finetune_QwenASR.git
!git -C revolab-asr-benchmark pull -q 2>/dev/null || git clone -q https://github.com/revolab-sdn-bhd/revolab-asr-benchmark.git
!pip install -q "qwen-asr==0.0.6" peft jiwer
# Colab ships torchao 0.10; current peft refuses any torchao < 0.16 when loading LoRA, and nothing here uses it
!pip uninstall -y -q torchao
import importlib, importlib.util
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao still installed: restart the runtime and rerun from the top"
%cd /content/finetune_QwenASR
!rm -rf outputs && ln -s "$DRIVE/outputs" outputs && ls -l outputs

In [ ]:
# Hugging Face token: needed for the gated Revolab benchmark (Colab secret HF_TOKEN, else prompt)
import os
from getpass import getpass
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
login(token=token or getpass("HF token: "))

## 2. Data and model on local disk

In [ ]:
# the trainer memmaps audio.bin; reading it through the Drive FUSE mount is far too slow, so copy it locally (~10GB)
!mkdir -p /content/data && rsync -a --info=progress2 "$DRIVE/malay_50h_babel" /content/data/
DATA = "/content/data/malay_50h_babel"
!du -sh $DATA/*

In [ ]:
from huggingface_hub import snapshot_download
M = snapshot_download("Qwen/Qwen3-ASR-1.7B")
snapshot_download("Revolab/ASR-Benchmark-Public", repo_type="dataset")  # eval_revolab.py reads it from the HF cache
os.environ.update(M=M, DATA=DATA, R="lora-1.7b-malay50h-babel", REVOLAB_DIR="/content/revolab-asr-benchmark",
                  PYTHONUTF8="1", USE_TF="0", TRANSFORMERS_NO_TF="1")
print(M)

## 3. Train (resumes from the latest checkpoint on Drive)

Same recipe as the local run; `--batch_size` must stay 4 or the resume is refused.

In [ ]:
import importlib.util
assert importlib.util.find_spec("torchao") is None, "torchao present: run the install cell (section 1) first"
!ls outputs/$R  # should list the latest checkpoint-* to resume from
!python -u train_qwen_asr.py --mode lora --model_path "$M" --lang Malay --data $DATA   --output_dir outputs/$R --epochs 2 --lr 1e-4 --batch_size 4 --grad_acc 4   --eval_steps 200 --save_steps 200 --workers 2 2>&1 | tee -a outputs/$R.colab.log | grep --line-buffered -E "resume|eval_loss|Error|Traceback|File \"|step.: [0-9]*00,"
!ls outputs/$R && cat outputs/$R/summary.json  # no summary.json = training did not finish: read outputs/$R.colab.log

## 4. Evaluate

In [ ]:
# in-domain test (malaya-stt, same 1,994 clips as malay_50h/test)
!python eval_wer.py --model_path "$M" --data $DATA --lang Malay --skip_base --runs $R --out outputs/eval_malay50h
!python score_malay.py --dir outputs/eval_malay50h

In [ ]:
# Revolab benchmark (820 clips, 12 domains)
!python eval_revolab.py --model_path "$M" --skip_base --runs $R --out outputs/eval_revolab

In [ ]:
# synthetic code-switch test
!python eval_wer.py --model_path "$M" --data $DATA --split cs_test --lang Malay --skip_base --runs $R --out outputs/eval_cs_test

In [ ]:
# real code-switch test: held-out BabelSpeech speaker spk001.
# base and lora-mix were already scored on Drive (outputs/eval_babel_test/summary.json merges), so only this run is new;
# drop --skip_base and add lora-1.7b-malay50h-mix to --runs to redo them on a fresh Drive.
!python eval_wer.py --model_path "$M" --data $DATA --split babel_test --lang Malay --skip_base --runs $R --out outputs/eval_babel_test
!python score_malay.py --dir outputs/eval_babel_test
